In [9]:
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command


model=init_chat_model("google_genai:gemini-3.5-flash-lite")

class State(TypedDict):
    email:str
    status:str
    working:str

def approval_node(state: State):
    decision = interrupt({
        "question": "Do you approve sending this email?",
        "email": state["email"]
    })

    if decision=="approve":
        return {"status":"approved"}
    else:
        return {"status":"rejected"}


def send_email(state:State):
    if state["status"]=="approved":
        print(f"email is sent to {state["email"]}")
        return {"status":"sent","working":"yes"}
    return {"status":"Not sent"}


def decide_next_node(state):
    if state["status"]=="approved":
        return "sendEmail"
    else:
        return END

graph=StateGraph(State)

graph.add_node("approval",approval_node)
graph.add_node("sendEmail",send_email)

graph.add_edge(START,"approval")
graph.add_conditional_edges("approval",decide_next_node)
graph.add_edge("sendEmail",END)
memory=InMemorySaver()
graph=graph.compile(checkpointer=memory)
config = {
    "configurable": {
        "thread_id": "email-1"
    }
}
result = graph.invoke(
    {
        "email": "div@rem.com",
        "status": "pending"
    },
    config=config
)

print(result["__interrupt__"][0].value["question"])



Do you approve sending this email?


In [10]:
result=graph.invoke(Command(resume="reject"),config=config)
print(result)

{'email': 'div@rem.com', 'status': 'rejected'}
